# AML LoT - SOB and episode

Split from `AML_LOT_VAL.ipynb`. Fully commented-out cells removed.

# SPARK SESSION

In [ ]:
import findspark
import pandas as pd
findspark.init ()
import pyspark
from pyspark.sql import SparkSession

spark= SparkSession.builder\
              .config('spark.executor.instances','20')\
              .config('spark.driver.memory','5g')\
              .config('spark.executor.memory','20g')\
              .config('spark.executor.cores','5')\
              .config('spark.executor.memoryOverhead','4g')\
              .appName("AML_KA")\
              .master("yarn")\
              .getOrCreate()

In [ ]:
spark.sparkContext.addPyFile("pldlib.egg")
from pldlib import stencil

# GLOBAL VARIABLES

In [ ]:
cws = 'Z_ABV_CWS_MABI_ONC_ANALYTICS' 

# HCP-CHILD
reltio_tbl = 'abv_mhcd_synd.mhcd_rltn_all_org_customr_wkly_tbl'
heme_account='{cws}.heme_account'.format(cws=cws) # UPLOADED BY COE

tx_tbl_name = '{cws}.MABI_AML_REGIMEN_REVAMP_UPDATED_TX_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_tbl_final_name = '{cws}.MABI_AML_REGIMEN_REVAMP_UPDATED_TX_TBL_FINAL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

sob_tbl_name = '{cws}.MABI_AML_REGIMEN_REVAMP_SOB_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
episode_tbl_name = '{cws}.MABI_AML_REGIMEN_REVAMP_EPISODE_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
regimen_tbl_name = '{cws}.MABI_AML_REGIMEN_REVAMP_BASIC_REGIMEN_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

patient_group = '{cws}.MABI_AML_REGIMEN_REVAMP_PATIENT_COHORT_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

# LoT IC INELIGIBLE
ic_inelig_final = '{cws}.MABI_AML_IC_INELIG_LOT_FINAL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

# LOT IC ELIGIBLE
ic_elig_final = '{cws}.MABI_AML_IC_ELIG_LOT_FINAL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws) # LOT table IC ELIG
pat_comb_lot = '{cws}.MABI_AML_PATS_COMB_LOT_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)  # UNION OF BOTH LOT FINAL TABLES
pat_comb_lot_rr = '{cws}.MABI_AML_PATS_LOT_RR_12M_UPD_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)    

pat_fst_dx_tx = '{cws}.MABI_AML_LOT_PAT_FST_DX_TX_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

dx_lookback_elig = '{cws}.MABI_AML_LOT_PAT_DX_LB_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_lookback_elig = '{cws}.MABI_AML_LOT_PAT_TX_MX_LB_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_lookback_elig_rx = '{cws}.MABI_AML_LOT_PAT_TX_RX_LB_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_txl_mx_elig = '{cws}.MABI_AML_LOT_PAT_TX_MX_TXL_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_txl_rx_elig = '{cws}.MABI_AML_LOT_PAT_TX_RX_TXL_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

tx_lookback_elig_ven = '{cws}.MABI_AML_LOT_PAT_VEN_TX_MX_LB_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_lookback_elig_rx_ven = '{cws}.MABI_AML_LOT_PAT_VEN_TX_RX_LB_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_txl_mx_elig_ven = '{cws}.MABI_AML_LOT_PAT_VEN_TX_MX_TXL_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_txl_rx_elig_ven = '{cws}.MABI_AML_LOT_PAT_VEN_TX_RX_TXL_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

lot_regimen_group = '{cws}.MABI_AML_PATS_LOT_GROUPING_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
patient_intensity_based_on_age_n_product = '{cws}.PATIENT_INTENSITY_BASED_ON_AGE_N_PRODUCT_v2'.format(cws=cws)

#HCP AFFILIATIONS
hcp_affiliations = '{cws}.MABI_AML_HCP_AFFILIATIONS'.format(cws=cws)

# LoT TABLE ELIGIBILITY CHECK
elig_flags_final='{cws}.MABI_AML_LOT_PAT_ELIG_FLAGS_FINAL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

#RR AML DX ONLY
aml_dx = ('C92.01', 'C92.02', 'C92.51', 'C92.52', 'C92.61', 'C92.62', 'C92.A1', 'C92.A2', 'C93.01', 'C93.02', 'C94.01', 'C94.21', 'C94.22')


# REGIMEN CREATION

#### SOB

In [ ]:
sob_tbl_name

In [ ]:
sob_df = stencil.init(spark).sob(
    grace='GRACE_VALUE'
	, lookback='360'
	, dedup_type_vl='yesremove' #to handle multiple claims for a product on same day
	, dedup_partition_on_clmns='PATIENT_GID, FINAL_PRODUCT_NAME, CLAIM_DATE'
	, dedup_ranking_order_clmns='DOS_FINAL DESC, CLAIM_ID' #choose claim with max DOS among multiple claims for a product on same day
	, dos_clmn='DOS_FINAL'
	, patient_gid_clmn='PATIENT_GID'
	, claim_id_clmn='CLAIM_ID'
	, claim_start_dt_clmn='CLAIM_DATE'
	, product_name_clmn='FINAL_PRODUCT_NAME'
	, src_input_tbl='(SELECT *, "202210" AS DATA_PERIOD FROM {tx_tbl_final_name})'.format(tx_tbl_final_name=tx_tbl_final_name)
	, metric_ordering_by_clmns='CLAIM_DATE, CLAIM_ID'
	, metric_ordering_by_following_clmns='CLAIM_DATE, CLAIM_ID'
	, dos_metric_partition_on_clmns='PATIENT_GID, FINAL_PRODUCT_NAME, CLAIM_DATE'
	, data_end_required='no'
	, required_grace='0'
)

sob_df.createOrReplaceTempView('sob_df')

In [ ]:
spark.sql('''DROP TABLE IF EXISTS {sob_tbl_name}'''.format(sob_tbl_name=sob_tbl_name))

In [ ]:
spark.sql('''
CREATE TABLE {sob_tbl_name} AS
(
    SELECT *
    FROM sob_df
)
'''.format(sob_tbl_name=sob_tbl_name))

In [ ]:
spark.sql('''
SELECT COUNT(DISTINCT PATIENT_GID) AS NUM_PAT
FROM {sob_tbl_name}
'''.format(sob_tbl_name=sob_tbl_name)).toPandas().head(20)

#### EPISODE

In [ ]:
episode_tbl_name

In [ ]:
episode_df = stencil.init(spark).episode(
	src_input_tbl='sob_df'
	, patient_gid_clmn='PATIENT_GID'
	, product_name_clmn='FINAL_PRODUCT_NAME'
	, metric_ordering_by_clmns='CLAIM_DATE, CLAIM_ID'
	, claim_rnkn_partition_on_clmns='PATIENT_GID, FINAL_PRODUCT_NAME, EPISODE_NUM_DRVD'
	, claim_rnkn_ordering_by_clmns='CLAIM_DATE, CLAIM_ID'
	, claim_id_clmn='CLAIM_ID'
	, episode_end_dts_clmns='EPISODE_END_DATE1_DRVD, EPISODE_END_DATE2_DRVD, EPISODE_END_DATE3_DRVD'
	, claim_start_dt_clmn='CLAIM_DATE'
	, drug_class_clmn ='"NA"'
	, dos_null_flag_clmns='DOS_NULL_FLAG, DOS_PAT_PRD_NULL_FLAG, DOS_PAT_PRD_EPSD_NULL_FLAG'
)

episode_df.createOrReplaceTempView('episode_df')

In [ ]:
spark.sql('''DROP TABLE IF EXISTS {episode_tbl_name}'''.format(episode_tbl_name=episode_tbl_name))

In [ ]:
spark.sql('''
CREATE TABLE {episode_tbl_name} AS
(
    SELECT *
    FROM episode_df
)
'''.format(episode_tbl_name=episode_tbl_name))

In [ ]:
spark.sql('''
SELECT COUNT(DISTINCT PATIENT_GID) AS NUM_PAT
FROM {episode_tbl_name}
'''.format(episode_tbl_name=episode_tbl_name)).toPandas().head(20)